In [5]:
import sys
from pathlib import Path

_REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(_REPO_ROOT))
from core.cdse import find_l1c, sensing_and_tile

import argparse
import json
import os
import shutil
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import pystac
import requests
import xarray as xr
from dotenv import load_dotenv
from pyproj import Transformer
from tqdm.auto import tqdm

In [2]:
matchups = gpd.read_file("../../data/nuevos/matchups_registros_finales.geojson")

In [3]:
matchups.head(2)

,fecha,fuente,chla,grupo_nombre,estado_trofico,pass_id,delta,geometry
0,2017-01-02,OAN,8.9,RDP-MONTES,M,S2A_MSIL1C_20170104T135052_N0500_R024_T21HUC_2...,222652.026,POINT (401164.986 6214585.025)
1,2017-01-02,GEMS,6.8,LDS,O,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,296857.974,POINT (678380.002 6147956.963)


In [11]:
CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
ITEMS_FILE = "../../data/nuevos/matchup_items_nuevos.json"
ZIPPER = "https://zipper.dataspace.copernicus.eu/odata/v1/Products({id})/$value"

In [12]:
items = {it.id: it for it in pystac.ItemCollection.from_file(str(ITEMS_FILE))}

In [15]:
list(items.keys())[0]

'S2B_MSIL1C_20171105T135059_N0500_R024_T21HUD_20231013T190243'

## Estructura de los IDs de productos Sentinel-2

`S2B_MSIL1C_20171105T135059_N0500_R024_T21HUD_20231013T190243`

Cada campo va separado por `_` y sigue la convención oficial de ESA:

| # | Campo | Ejemplo | Significado |
|---|-------|---------|-------------|
| 1 | Satélite | `S2B` | Sentinel-2B (puede ser `S2A`, `S2B` o `S2C`) |
| 2 | Instrumento y nivel | `MSIL1C` | MultiSpectral Instrument, nivel L1C (reflectancia de tope de atmósfera). `MSIL2A` ya trae corrección atmosférica de Sen2Cor |
| 3 | Adquisición | `20171105T135059` | Fecha y hora UTC de inicio de la toma: 05/11/2017 13:50:59 UTC (10:50 en Uruguay) |
| 4 | Baseline de procesamiento | `N0500` | Versión del procesador de ESA (05.00) |
| 5 | Órbita relativa | `R024` | Traza del satélite (1–143). Un tile puede verse desde dos órbitas con geometrías distintas |
| 6 | Tile MGRS | `T21HUD` | Cuadrícula de ~110 × 110 km: zona UTM `21`, banda de latitud `H`, cuadrado de 100 km `UD` (→ EPSG:32721) |
| 7 | Discriminador del producto | `20231013T190243` | Fecha de generación del producto. Distingue versiones de la misma escena |

### Reprocesamiento (Collection-1)

En este ejemplo, la imagen es de **2017**, pero el producto se generó en **2023** con el baseline **N0500**. Es parte del **reprocesamiento de la Collection-1** de ESA: el archivo histórico se volvió a procesar con calibración y georreferenciación mejoradas y consistentes.

- Una misma adquisición puede aparecer varias veces en CDSE (por ejemplo `N0206` de 2017 y `N0500` de 2023).
- Para series temporales conviene usar la versión reprocesada, para que todas las fechas estén procesadas igual.
- `find_l1c()` ordena por baseline y discriminador y se queda con la más reciente.

### Relación con los `pass_id` del STAC

Los IDs de Planetary Computer no incluyen el baseline, por ejemplo:

`S2B_MSIL2A_20171105T135059_R024_T21HUD_20201014T...`

Para emparejar el L2A del STAC con el L1C de CDSE se usan:

- la **fecha y hora de adquisición** (campo 3)
- el **tile** (campo 6)

In [23]:
_id = list(items.keys())[0]
fecha, tile = sensing_and_tile(items[_id])
fecha, tile


('20171105T135059', '21HUD')

In [25]:
product = find_l1c(fecha, tile)

In [26]:
product

{'name': 'S2B_MSIL1C_20171105T135059_N0500_R024_T21HUD_20231013T190243.SAFE',
 'id': 'abad1d8b-9507-4cd6-ae59-1d1bcbfca6e4',
 'online': True}

In [27]:
products = []

In [28]:
for pass_id in tqdm(sorted(matchups["pass_id"].unique()), desc="Matching L1C"):
    #if pass_id in plan_data:
        #continue
        
    item = items.get(pass_id)
    if item is None:
        print(f"[{pass_id}] not in {ITEMS_FILE.name}, skipping")
        continue
    # ACA: 
    stamp, tile = sensing_and_tile(item)
    try:
        l1c = find_l1c(stamp, tile)
    except requests.RequestException as e:
        print(f"[{pass_id}] CDSE query failed: {e}")
        continue
    if l1c is None:
        print(f"[{pass_id}] no L1C found for {stamp} T{tile}")
        
    products.append(l1c)

Matching L1C: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 529/529 [1:04:09<00:00,  7.28s/it]


In [32]:
products_df = pd.DataFrame(products)
products_df.head(2)

,name,id,online
0,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,d4d98e81-e795-4059-861d-611143221276,True
1,S2A_MSIL1C_20170104T135052_N0500_R024_T21HUC_2...,1963fc63-9e81-4cbc-a27a-5f951e6c280b,True


In [35]:
products_df.to_csv("../../data/nuevos/l1c_scene_products.csv", index=False)